# CLM Upcycling Study 001 — Inherit Then Differentiate
Exact dense-FFN inheritance versus local-state-geometry router initialization.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
ROOT = Path('/kaggle/working/mini-cells')
REF = os.environ.get('MINICELLS_REF', 'research/clm-upcycling-study-001')
os.chdir('/kaggle/working')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth','1','--branch',REF,
                'https://github.com/ArcheLabs/mini-cells.git',str(ROOT)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[lm,dev]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print(subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip())


In [ ]:
import torch, platform
print({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu_count': torch.cuda.device_count(),
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
})
assert torch.cuda.is_available()
for path in [
    'src/minicells/upcycled_cellular_textnca.py',
    'src/minicells/clm_upcycling_validation.py',
    'scripts/run_clm_upcycling_study_001_worker.py',
    'scripts/run_clm_upcycling_study_001.py',
    'scripts/publish_clm_upcycling_study_001_results.py',
]:
    subprocess.run([sys.executable,'-m','py_compile',path], cwd=ROOT, check=True)
subprocess.run([
    sys.executable,'-m','pytest',
    'tests/test_clm_upcycling_study_001.py',
    'tests/test_clm_v2_validation.py',
    'tests/test_overcomplete_cellular_textnca.py',
    '-q',
], cwd=ROOT, check=True)


In [ ]:
subprocess.run(
    [sys.executable, 'scripts/run_clm_upcycling_study_001.py'],
    cwd=ROOT,
    check=True,
)


In [ ]:
import json, pandas as pd
from IPython.display import Image, display
OUT = ROOT / 'results' / 'clm-upcycling-study-001-inherit-then-differentiate'
print(json.dumps(json.loads((OUT/'decision.json').read_text()), indent=2))
display(pd.read_csv(OUT/'progression.csv'))
display(pd.read_csv(OUT/'controls.csv'))
for name in [
    'continuation-quality.png',
    'final-quality.png',
    'expert-divergence.png',
    'usage-entropy.png',
    'routing-controls.png',
    'routing-variation.png',
    'capacity-vs-active.png',
]:
    display(Image(filename=str(OUT/name)))


In [ ]:
PUBLISH = False
if PUBLISH:
    subprocess.run(
        [sys.executable, 'scripts/publish_clm_upcycling_study_001_results.py', '--push'],
        cwd=ROOT,
        check=True,
    )
else:
    print('PUBLISH=False; inspect decision and plots before publishing.')
